# Component 1 — Aspect-Conditioned Sentiment and Confidence Calibration

In [ ]:

from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

CURRENT = Path.cwd().resolve()
if CURRENT.name == "notebooks":
    PROJECT_ROOT = CURRENT.parent
elif (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise RuntimeError(f"Could not locate component_1 from {CURRENT}")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mabsa_aspect_sentiment.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "sentiment"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 224
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
SMOKE_TEST = False
SAVE_MODEL_WEIGHTS = False
CALIBRATION_STEPS = 300
CALIBRATION_LR = 0.03

SENTIMENT_TO_ID = {"negative": 0, "neutral": 1, "positive": 2}
ID_TO_SENTIMENT = {v: k for k, v in SENTIMENT_TO_ID.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)


In [ ]:

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

data = pd.read_csv(DATA_PATH)

required_columns = {
    "review_id",
    "domain",
    "split",
    "text",
    "canonical_aspect",
    "semantic_group",
    "sentiment",
    "is_null_aspect_term",
}
missing = required_columns - set(data.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

data["sentiment"] = data["sentiment"].str.lower().str.strip()
data["sentiment_id"] = data["sentiment"].map(SENTIMENT_TO_ID)

if data["sentiment_id"].isna().any():
    raise ValueError("Unexpected sentiment label found.")

train_df = data[data["split"].eq("train")].reset_index(drop=True)
dev_df = data[data["split"].eq("dev")].reset_index(drop=True)
test_df = data[data["split"].eq("test")].reset_index(drop=True)

if SMOKE_TEST:
    train_df = train_df.head(256)
    dev_df = dev_df.head(128)
    test_df = test_df.head(128)

display(
    data.groupby(["domain", "split", "sentiment"])
    .size()
    .rename("count")
    .reset_index()
)


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class AspectSentimentDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

        aspect_prompts = (
            "Aspect: " + self.df["canonical_aspect"].astype(str)
        ).tolist()

        self.enc = tokenizer(
            self.df["text"].astype(str).tolist(),
            aspect_prompts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

        self.labels = self.df["sentiment_id"].astype(int).to_numpy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def make_loader(df, shuffle=False):
    return DataLoader(
        AspectSentimentDataset(df),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


train_loader = make_loader(train_df, shuffle=True)
dev_loader = make_loader(dev_df)
test_loader = make_loader(test_df)


In [ ]:

class_counts = train_df["sentiment_id"].value_counts().sort_index()

weights = np.array([
    len(train_df) / (len(SENTIMENT_TO_ID) * class_counts.get(i, 1))
    for i in range(len(SENTIMENT_TO_ID))
], dtype=np.float32)

class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=SENTIMENT_TO_ID,
    id2label=ID_TO_SENTIMENT,
).to(DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)

criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

history = []
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for batch in tqdm(train_loader, desc=f"sentiment epoch {epoch}/{EPOCHS}"):
        labels = batch["labels"].to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}

        optimizer.zero_grad()
        logits = model(**inputs).logits
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())

    model.eval()
    dev_losses = []

    with torch.no_grad():
        for batch in dev_loader:
            labels = batch["labels"].to(DEVICE)
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            logits = model(**inputs).logits
            dev_losses.append(criterion(logits, labels).item())

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "dev_loss": float(np.mean(dev_losses)),
    }
    history.append(row)
    print(row)


In [ ]:

def collect_logits(loader):
    model.eval()
    logits_list, labels_list = [], []

    with torch.no_grad():
        for batch in loader:
            labels_list.append(batch["labels"].numpy())
            inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            logits_list.append(model(**inputs).logits.cpu().numpy())

    return np.vstack(logits_list), np.concatenate(labels_list)


dev_logits_np, dev_labels = collect_logits(dev_loader)
test_logits_np, test_labels = collect_logits(test_loader)

dev_logits = torch.tensor(dev_logits_np, dtype=torch.float32)
dev_targets = torch.tensor(dev_labels, dtype=torch.long)

log_temperature = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)
temp_optimizer = torch.optim.Adam([log_temperature], lr=CALIBRATION_LR)

for _ in range(CALIBRATION_STEPS):
    temp_optimizer.zero_grad()
    temperature_tensor = torch.exp(log_temperature).clamp(0.05, 10.0)
    loss = torch.nn.functional.cross_entropy(
        dev_logits / temperature_tensor,
        dev_targets,
    )
    loss.backward()
    temp_optimizer.step()

temperature = float(
    torch.exp(log_temperature).clamp(0.05, 10.0).detach().item()
)

print("Learned temperature:", round(temperature, 4))


In [ ]:

def softmax_np(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def multiclass_brier(y_true, probs, num_classes=3):
    one_hot = np.eye(num_classes)[y_true]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def expected_calibration_error(y_true, probs, n_bins=10):
    confidence = probs.max(axis=1)
    prediction = probs.argmax(axis=1)
    correct = (prediction == y_true).astype(float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (confidence >= edges[i]) & (confidence <= edges[i + 1])
        else:
            mask = (confidence >= edges[i]) & (confidence < edges[i + 1])

        if not mask.any():
            continue

        bin_acc = correct[mask].mean()
        bin_conf = confidence[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)

    return float(ece)


def evaluate_probabilities(y_true, probs):
    pred = probs.argmax(axis=1)

    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "nll": float(log_loss(y_true, probs, labels=[0, 1, 2])),
        "brier": multiclass_brier(y_true, probs),
        "ece": expected_calibration_error(y_true, probs),
        "mean_confidence": float(probs.max(axis=1).mean()),
    }, pred


uncalibrated_probs = softmax_np(test_logits_np)
calibrated_probs = softmax_np(test_logits_np / temperature)

uncal_metrics, uncal_pred = evaluate_probabilities(
    test_labels,
    uncalibrated_probs,
)
cal_metrics, cal_pred = evaluate_probabilities(
    test_labels,
    calibrated_probs,
)

metric_rows = [
    {"segment": "overall", "calibration": "uncalibrated", **uncal_metrics},
    {"segment": "overall", "calibration": "calibrated", **cal_metrics},
]

for domain in ["phone", "laptop"]:
    idx = test_df["domain"].eq(domain).to_numpy()

    m1, _ = evaluate_probabilities(
        test_labels[idx],
        uncalibrated_probs[idx],
    )
    m2, _ = evaluate_probabilities(
        test_labels[idx],
        calibrated_probs[idx],
    )

    metric_rows.extend([
        {"segment": domain, "calibration": "uncalibrated", **m1},
        {"segment": domain, "calibration": "calibrated", **m2},
    ])

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)


In [ ]:

predictions_df = test_df[
    [
        "review_id",
        "domain",
        "text",
        "canonical_aspect",
        "semantic_group",
        "sentiment",
        "is_null_aspect_term",
    ]
].copy()

predictions_df["true_sentiment"] = [
    ID_TO_SENTIMENT[int(x)] for x in test_labels
]
predictions_df["predicted_sentiment_uncalibrated"] = [
    ID_TO_SENTIMENT[int(x)] for x in uncal_pred
]
predictions_df["predicted_sentiment_calibrated"] = [
    ID_TO_SENTIMENT[int(x)] for x in cal_pred
]
predictions_df["confidence_uncalibrated"] = (
    uncalibrated_probs.max(axis=1)
)
predictions_df["confidence_calibrated"] = (
    calibrated_probs.max(axis=1)
)

cm = confusion_matrix(
    test_labels,
    cal_pred,
    labels=[0, 1, 2],
)

confusion_df = pd.DataFrame(
    cm,
    index=["true_negative", "true_neutral", "true_positive"],
    columns=["pred_negative", "pred_neutral", "pred_positive"],
)

history_df = pd.DataFrame(history)

metrics_df.to_csv(
    RESULTS_DIR / "sentiment_calibration_metrics.csv",
    index=False,
)
predictions_df.to_csv(
    RESULTS_DIR / "sentiment_test_predictions.csv",
    index=False,
)
history_df.to_csv(
    RESULTS_DIR / "sentiment_training_history.csv",
    index=False,
)
confusion_df.to_csv(
    RESULTS_DIR / "sentiment_confusion_matrix.csv"
)

with open(
    RESULTS_DIR / "sentiment_run_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "seed": SEED,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "temperature": temperature,
            "smoke_test": SMOKE_TEST,
            "save_model_weights": SAVE_MODEL_WEIGHTS,
            "device": str(DEVICE),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

if SAVE_MODEL_WEIGHTS:
    torch.save(
        model.state_dict(),
        RESULTS_DIR / "sentiment_roberta_state_dict.pt",
    )

display(confusion_df)


In [ ]:

required_outputs = [
    RESULTS_DIR / "sentiment_calibration_metrics.csv",
    RESULTS_DIR / "sentiment_test_predictions.csv",
    RESULTS_DIR / "sentiment_training_history.csv",
    RESULTS_DIR / "sentiment_confusion_matrix.csv",
    RESULTS_DIR / "sentiment_run_config.json",
]

checks = {
    "temperature_positive": temperature > 0,
    "metrics_valid": metrics_df[
        ["accuracy", "macro_f1", "ece", "brier"]
    ].notna().all().all(),
    "predictions_match_test_size": len(predictions_df) == len(test_df),
    "outputs_saved": all(path.exists() for path in required_outputs),
}

validation_df = pd.DataFrame(
    [{"check": k, "passed": bool(v)} for k, v in checks.items()]
)

display(validation_df)

if not validation_df["passed"].all():
    failed = validation_df.loc[
        ~validation_df["passed"],
        "check",
    ].tolist()
    raise ValueError(
        f"SENTIMENT/CALIBRATION VALIDATION FAILED: {failed}"
    )

print("C1 ASPECT-CONDITIONED SENTIMENT + CALIBRATION: PASSED")
